<a href="https://colab.research.google.com/github/Sharma-Himansh/Data-Science-Visualisation-Lab/blob/main/BCSE1593_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCSE 1593 – Data Science and Visualization Lab

## Practical: Data Wrangling

In this practical, we will take a retail transaction dataset and reorganise it into forms that are easier to analyse, compare and report.

The work will follow a simple sequence:

> **Load → understand the structure → create hierarchical indexes → combine tables → merge related data → reshape → verify → build a final report**

We will work with the **Supermart Grocery Sales – Retail Analytics Dataset**, a Kaggle retail dataset containing order, customer, category, region, date, sales, discount and profit information.

### In this practical, we will learn to

- create and work with a **MultiIndex**
- select records from specific index levels
- reorder and sort hierarchical indexes
- combine tables using `concat()`
- perform database-style joins using `merge()`
- merge tables using index columns
- use `join()` for index-based combination
- reshape data from wide to long using `melt()`
- reshape data from long to wide using `pivot()` and `pivot_table()`
- create and work with hierarchical columns
- use `stack()` and `unstack()`
- use `wide_to_long()` for repeated column patterns
- verify the structure after each major transformation

The focus is on understanding **how data changes shape** and choosing the correct pandas operation for the task.

# Part A – Getting Started

## 1. Importing pandas

Pandas will be our main tool for working with tabular data, indexes, joins and reshaping.

### Live coding

Import pandas using the standard alias `pd`.

In [1]:
print("Importing pandas")

# Import pandas using the standard alias pd.
import pandas as pd

Importing pandas


## 2. Importing NumPy

NumPy is useful for small helper tables and numerical operations during the practical.

### Live coding

Import NumPy using the standard alias `np`.

In [2]:
print("Importing NumPy")

# Import NumPy using the standard alias np.
import numpy as np

Importing NumPy


# Part B – Loading the Dataset

## 3. Dataset source – Supermart Grocery Sales

The dataset used here is **Supermart Grocery Sales – Retail Analytics Dataset**, available on Kaggle.

Kaggle source:

**https://www.kaggle.com/datasets/mohamedharris/supermart-grocery-sales-retail-analytics-dataset**

The dataset contains retail order records such as `Order ID`, `Customer Name`, `Category`, `Sub Category`, `City`, `Order Date`, `Region`, `Sales`, `Discount`, and `Profit`.

### Getting the file

Download **Supermart Grocery Sales - Retail Analytics Dataset.csv** from Kaggle and keep it in the same folder as this notebook.

The loading cell below first tries a local file. A public copy of the same CSV is provided as a fallback for practice environments where the local file is not available. **The dataset source remains Kaggle.**

In [3]:
file_name = "Supermart Grocery Sales - Retail Analytics Dataset.csv"

try:
    sales = pd.read_csv(file_name)
except FileNotFoundError:
    url = "https://raw.githubusercontent.com/godwinnwalozie/Supermart-Grocery-Sales---Retail-Analytics-Dataset/main/Supermart%20Grocery%20Sales%20-%20Retail%20Analytics%20Dataset.csv"
    sales = pd.read_csv(url)

print("Dataset loaded successfully")
sales.head()

Dataset loaded successfully


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,11-08-2017,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,11-08-2017,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,06-12-2017,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,10-11-2016,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,10-11-2016,South,2355,0.26,918.45,Tamil Nadu


## 4. First look at the data

Before changing the structure, first understand what a row represents and which columns are available.

Check the shape, column names, data types and a quick statistical summary.

In [4]:
# Display the number of rows and columns.
print('Shape (rows, columns):', sales.shape)
# Display the column names.
print('\nColumns:', sales.columns.tolist())

# Display the data types.
print('\nData types:\n', sales.dtypes)

# Display the first five rows.
display(sales.head())

# Display the numerical summary.
display(sales.describe(include='number'))


Shape (rows, columns): (9994, 11)

Columns: ['Order ID', 'Customer Name', 'Category', 'Sub Category', 'City', 'Order Date', 'Region', 'Sales', 'Discount', 'Profit', 'State']

Data types:
 Order ID          object
Customer Name     object
Category          object
Sub Category      object
City              object
Order Date        object
Region            object
Sales              int64
Discount         float64
Profit           float64
State             object
dtype: object


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,11-08-2017,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,11-08-2017,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,06-12-2017,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,10-11-2016,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,10-11-2016,South,2355,0.26,918.45,Tamil Nadu


,Sales,Discount,Profit
count,9994.000000,9994.000000,9994.000000
mean,1496.596158,0.226817,374.937082
std,577.559036,0.074636,239.932881
min,500.000000,0.100000,25.250000
25%,1000.000000,0.160000,180.022500
50%,1498.000000,0.230000,320.780000
75%,1994.750000,0.290000,525.627500
max,2500.000000,0.350000,1120.950000


## 5. Prepare the date column

Wrangling often becomes easier after dates are stored as real datetime values. This lets us create year, month and other time-based fields later.

### Function used

`pd.to_datetime()` converts text values into pandas datetime values.

The argument `errors='coerce'` turns an unrecognised date into `NaT` instead of stopping the program.

In [5]:
# Convert Order Date to datetime.
sales['Order Date'] = pd.to_datetime(sales['Order Date'], errors='coerce')

# Check the dtype of Order Date.
sales['Order Date'].dtype

dtype('<M8[ns]')

## 6. Create a working copy

Keep the original table unchanged and perform wrangling on a working copy. This is useful when you need to compare the raw and transformed versions later.

In [6]:
# Create a working copy named wrangled.
wrangled = sales.copy()

# Display the shape of wrangled.
print('Working table shape:', wrangled.shape)


Working table shape: (9994, 11)


# Part C – Hierarchical Indexing (MultiIndex)

## 7. Why use a hierarchical index?

A normal DataFrame uses one index for rows. A **MultiIndex** uses two or more levels.

For example, retail data can naturally be organised as:

> **Region → Category → records**

This is useful when the same type of question is repeatedly asked at more than one level, such as regional and category-wise reporting.

### Why is this useful in practice?

A sales team may want to move from **Region** to **Category** without creating separate tables for every combination. A MultiIndex keeps that relationship organised in one structure and makes repeated selections and summaries easier.

### Main functions

- `set_index()` – move columns into the index
- `pd.MultiIndex.from_arrays()` – create levels directly
- `pd.MultiIndex.from_tuples()` – create levels from tuples
- `.loc[]` – select using index labels
- `xs()` – select a cross-section
- `swaplevel()` – exchange level positions
- `reorder_levels()` – choose a new level order
- `sort_index()` – sort the index
- `reset_index()` – return index levels to columns


## 8. Creating a MultiIndex with `set_index()`

Move `Region` and `Category` into the index.

The original columns are not lost; they are simply moved into the row index.

This creates a structure such as **Region → Category → rows**.

In [7]:
# Keep a few useful columns for the MultiIndex example.
mi_data = wrangled[['Order ID', 'Customer Name', 'Region', 'Category', 'Sales', 'Profit']].copy()

# Set Region and Category as the two index levels.
# Store the result as region_category_mi.
region_category_mi = mi_data.set_index(['Region', 'Category'])
# Display the first few rows.
display(region_category_mi.head())


Order ID Customer Name  Sales  Profit
Region Category                                              
North  Oil & Masala          OD1        Harish   1254  401.28
South  Beverages             OD2         Sudha    749  149.80
West   Food Grains           OD3       Hussain   2360  165.20
South  Fruits & Veggies      OD4       Jackson    896   89.60
       Food Grains           OD5       Ridhesh   2355  918.45

## 9. Inspecting the index levels

A MultiIndex contains multiple named levels. Inspecting them helps you understand how the table is organised.

Useful attributes and methods:

- `.index.names` – level names
- `.index.get_level_values()` – values from one level
- `.index.nlevels` – number of levels

In [8]:
# Display the names of the index levels.
print('Index level names:', region_category_mi.index.names)

# Display the number of index levels.
print('Number of index levels:', region_category_mi.index.nlevels)


# Display unique Region values from the first level.
print('Regions:', region_category_mi.index.get_level_values('Region').unique().tolist())

# Display unique Category values from the second level.
print('Categories:', region_category_mi.index.get_level_values('Category').unique().tolist())


Index level names: ['Region', 'Category']
Number of index levels: 2
Regions: ['North', 'South', 'West', 'Central', 'East']
Categories: ['Oil & Masala', 'Beverages', 'Food Grains', 'Fruits & Veggies', 'Bakery', 'Snacks', 'Eggs, Meat & Fish']


## 10. Selecting from a MultiIndex with `.loc`

When the DataFrame has meaningful row labels, `.loc` can select using those labels.

Here we will use the first available Region and Category from the data so that the example remains valid even if a student uses a filtered copy of the dataset.

In [9]:
# Pick one Region and one Category from the available values.
region_value = region_category_mi.index.get_level_values('Region').unique()[0]
category_value = region_category_mi.index.get_level_values('Category').unique()[0]

# Select rows matching both levels with .loc.
selected_pair = region_category_mi.loc[(region_value, category_value)]
print(f'Records for {region_value} / {category_value}:')
display(selected_pair)

Records for North / Oil & Masala:


/tmp/ipykernel_2089/1947259819.py:6: PerformanceWarning: indexing past lexsort depth may impact performance.
  selected_pair = region_category_mi.loc[(region_value, category_value)]


,,Order ID,Customer Name,Sales,Profit
Region,Category,,,,
North,Oil & Masala,OD1,Harish,1254,401.28


## 11. Selecting one level with `xs()`

The `xs()` method means **cross-section**. It is useful when you want to select one level without writing the complete index key.

Example uses include:

- all categories in one Region
- all Regions for one Category

In [10]:
# Select one Region using xs().
# Select one Category using xs(level='Category').
# xs() selects one cross-section of a MultiIndex.
selected_region = region_category_mi.xs(region_value, level='Region')
selected_category = region_category_mi.xs(category_value, level='Category')
print('Selected Region:', region_value)
display(selected_region.head())
print('Selected Category:', category_value)
display(selected_category.head())

Selected Region: North


,Order ID,Customer Name,Sales,Profit
Category,,,,
Oil & Masala,OD1,Harish,1254,401.28


Selected Category: Oil & Masala


,Order ID,Customer Name,Sales,Profit
Region,,,,
North,OD1,Harish,1254,401.28
Central,OD16,Ramesh,1440,100.80
East,OD30,Shah,989,69.23
East,OD33,Akash,2163,670.53
East,OD34,Anu,1689,337.80


## 12. Creating a MultiIndex explicitly with `from_arrays()`

A MultiIndex does not have to come from existing DataFrame columns. It can also be created directly from arrays.

This is useful when building small lookup tables or preparing a report structure.

In [11]:
regions = ['North', 'South', 'West', 'East']
categories = ['Bakery', 'Beverages', 'Snacks', 'Oil & Masala']
sales_values = [12000, 9500, 8300, 10600]

# Create a MultiIndex from the two arrays.
small_index = pd.MultiIndex.from_arrays([regions, categories])


# Give the levels the names Region and Category.
small_index = small_index.set_names(['Region', 'Category'])


# Build a small DataFrame named regional_sales.
regional_sales = pd.DataFrame({'Sales': sales_values}, index=small_index)
display(regional_sales)

,,Sales
Region,Category,
North,Bakery,12000
South,Beverages,9500
West,Snacks,8300
East,Oil & Masala,10600


## 13. Creating a MultiIndex with `from_tuples()`

`from_tuples()` is convenient when each index key is naturally written as a pair such as `(Region, Category)`.

In [12]:
keys = [('North', 'Bakery'), ('South', 'Beverages'), ('West', 'Snacks'), ('East', 'Oil & Masala')]

# Create a MultiIndex from the tuple pairs.
tuple_index = pd.MultiIndex.from_tuples(keys, names=['Region', 'Category'])


# Display the resulting MultiIndex.
print(tuple_index)

MultiIndex([('North',       'Bakery'),
            ('South',    'Beverages'),
            ( 'West',       'Snacks'),
            ( 'East', 'Oil & Masala')],
           names=['Region', 'Category'])


## 14. Reordering the index levels

The order of levels affects how the data is presented and selected.

Two useful functions are:

- `swaplevel()` – quickly exchange two levels
- `reorder_levels()` – specify the complete new order

In [13]:
# Swap Region and Category.
swapped_sales = regional_sales.swaplevel('Region', 'Category')

# Reorder the levels so Category comes before Region.
category_first = regional_sales.reorder_levels(['Category', 'Region'])

# Display both results.
print('After swaplevel:')
display(swapped_sales)
print('After reorder_levels:')
display(category_first)

After swaplevel:


,,Sales
Category,Region,
Bakery,North,12000
Beverages,South,9500
Snacks,West,8300
Oil & Masala,East,10600


After reorder_levels:


,,Sales
Category,Region,
Bakery,North,12000
Beverages,South,9500
Snacks,West,8300
Oil & Masala,East,10600


## 15. Sorting by index levels

After changing index levels, sorting keeps the hierarchy organised.

Use `sort_index()` for MultiIndex sorting. You can sort all levels or specify a particular level.

In [14]:
# Sort regional_sales by all index levels.
sorted_sales = regional_sales.sort_index()

# Sort another version primarily by Category.
sorted_by_category = regional_sales.sort_index(level='Category')

display(sorted_sales)
display(sorted_by_category)

,,Sales
Region,Category,
East,Oil & Masala,10600
North,Bakery,12000
South,Beverages,9500
West,Snacks,8300


,,Sales
Region,Category,
North,Bakery,12000
South,Beverages,9500
East,Oil & Masala,10600
West,Snacks,8300


## 16. Selecting a range with `IndexSlice`

`pd.IndexSlice` makes multi-level `.loc[]` expressions easier to read. It is helpful when you want several values from one level while keeping all values from another level.

In [15]:
# Create an IndexSlice object.
idx = pd.IndexSlice

# Use idx with .loc to select multiple Regions from region_category_mi.
# Keep all Category values.
selected_regions = region_category_mi.loc[idx[['North', 'South'], :], :]
display(selected_regions.head(10))

Order ID Customer Name  Sales   Profit
Region Category                                               
North  Oil & Masala          OD1        Harish   1254   401.28
South  Beverages             OD2         Sudha    749   149.80
       Fruits & Veggies      OD4       Jackson    896    89.60
       Food Grains           OD5       Ridhesh   2355   918.45
       Snacks               OD13        Sharon   1659   315.21
       Snacks               OD44        Mathew   2311  1039.95
       Oil & Masala         OD70          Peer   1232   554.40
       Snacks               OD73          Alan    721   100.94
       Fruits & Veggies     OD74         Vidya   2375   475.00
       Beverages            OD75        Arvind   1965   117.90

## 17. Resetting a MultiIndex

A hierarchical index is useful for analysis, but ordinary columns are often easier for merging, exporting and further transformation.

`reset_index()` moves index levels back into columns.

In [16]:
# Reset the MultiIndex.
mi_reset = region_category_mi.reset_index()

# Display the resulting columns.
print(mi_reset.columns.tolist())
display(mi_reset.head())


['Region', 'Category', 'Order ID', 'Customer Name', 'Sales', 'Profit']


,Region,Category,Order ID,Customer Name,Sales,Profit
0,North,Oil & Masala,OD1,Harish,1254,401.28
1,South,Beverages,OD2,Sudha,749,149.80
2,West,Food Grains,OD3,Hussain,2360,165.20
3,South,Fruits & Veggies,OD4,Jackson,896,89.60
4,South,Food Grains,OD5,Ridhesh,2355,918.45


## 18. MultiIndex created automatically by `groupby()`

Grouping by more than one column commonly creates a MultiIndex when `as_index=True`. This is one of the most useful real-world ways to encounter hierarchical indexes.

In [17]:
# Create a Region-Category summary.
# Calculate total Sales, total Profit and average Discount.
# Keep Region and Category as index levels.
region_category_summary = wrangled.groupby(['Region', 'Category']).agg(
    Total_Sales=('Sales', 'sum'),
    Total_Profit=('Profit', 'sum'),
    Average_Discount=('Discount', 'mean')
)

# Display the summary.
display(region_category_summary.head(10))


Total_Sales  Total_Profit  Average_Discount
Region  Category                                                      
Central Bakery                  448343     109543.91          0.231294
        Beverages               501194     128017.14          0.231749
        Eggs, Meat & Fish       516642     126587.35          0.232356
        Food Grains             463683     111517.05          0.223613
        Fruits & Veggies        526507     131699.53          0.232676
        Oil & Masala            453293     109261.91          0.224900
        Snacks                  558494     140179.95          0.224187
East    Bakery                  588241     153741.06          0.227908
        Beverages               577206     144498.17          0.230000
        Eggs, Meat & Fish       677830     164465.08          0.231180

# Part D – Combining Data Sets

## 19. Why combine tables?

Real projects rarely keep every piece of information in one table. You may receive files by month, by branch, or by business unit. Pandas provides several ways to bring them together.

### Why is this useful in practice?

A company may receive January, February and March sales files separately. The analyst needs one table before calculating totals, trends or reports. The correct combining method depends on whether the records should be **stacked**, **matched by a key**, or **aligned by index**.

### Main function

`pd.concat()` combines objects along rows or columns.

Use:

- `axis=0` for row-wise combination
- `axis=1` for column-wise combination
- `keys=` when you want to keep the source of each block


## 20. Preparing two related tables

To understand concatenation clearly, split the transaction table into two smaller batches. The records remain the same; only the storage is different.

In [18]:
# Create one table containing the first 5 rows.
batch_1 = wrangled.head(5).copy()

# Create another table containing the next 5 rows.
batch_2 = wrangled.iloc[5:10].copy()

# Display both batches.
display(batch_1)
display(batch_2)

,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,2017-11-08,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,2017-11-08,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,2017-06-12,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,2016-10-11,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,2016-10-11,South,2355,0.26,918.45,Tamil Nadu


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
5,OD6,Adavan,Food Grains,Organic Staples,Dharmapuri,2015-06-09,West,2305,0.26,322.70,Tamil Nadu
6,OD7,Jonas,Fruits & Veggies,Fresh Vegetables,Trichy,2015-06-09,West,826,0.33,346.92,Tamil Nadu
7,OD8,Hafiz,Fruits & Veggies,Fresh Fruits,Ramanadhapuram,2015-06-09,West,1847,0.32,147.76,Tamil Nadu
8,OD9,Hafiz,Bakery,Biscuits,Tirunelveli,2015-06-09,West,791,0.23,181.93,Tamil Nadu
9,OD10,Krithika,Bakery,Cakes,Chennai,2015-06-09,West,1795,0.27,484.65,Tamil Nadu


## 21. Vertical concatenation with `concat()`

When two tables have the same columns and represent the same kind of record, append one underneath the other with `pd.concat()`.

**Practical use:** this is common when monthly, branch-wise or batch-wise files have the same structure and need to become one analysis table.


In [19]:
# Concatenate batch_1 and batch_2 row-wise.
combined_rows = pd.concat([batch_1, batch_2], axis=0, ignore_index=True)

# Display the combined table.
display(combined_rows)

,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,2017-11-08,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,2017-11-08,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,2017-06-12,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,2016-10-11,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,2016-10-11,South,2355,0.26,918.45,Tamil Nadu
5,OD6,Adavan,Food Grains,Organic Staples,Dharmapuri,2015-06-09,West,2305,0.26,322.70,Tamil Nadu
6,OD7,Jonas,Fruits & Veggies,Fresh Vegetables,Trichy,2015-06-09,West,826,0.33,346.92,Tamil Nadu
7,OD8,Hafiz,Fruits & Veggies,Fresh Fruits,Ramanadhapuram,2015-06-09,West,1847,0.32,147.76,Tamil Nadu
8,OD9,Hafiz,Bakery,Biscuits,Tirunelveli,2015-06-09,West,791,0.23,181.93,Tamil Nadu
9,OD10,Krithika,Bakery,Cakes,Chennai,2015-06-09,West,1795,0.27,484.65,Tamil Nadu


## 22. Concatenation with source labels

`keys=` adds another index level showing where each block came from. This creates a useful hierarchical structure while combining data.

In [20]:
# Concatenate the two batches with source labels Batch_1 and Batch_2.
combined_with_keys = pd.concat([batch_1, batch_2], keys=['Batch_1', 'Batch_2'], names=['Source', 'Original_Row'])
# Display the result and inspect its MultiIndex.
display(combined_with_keys)
print('Index names:', combined_with_keys.index.names)

Order ID Customer Name          Category  \
Source  Original_Row                                            
Batch_1 0                 OD1        Harish      Oil & Masala   
        1                 OD2         Sudha         Beverages   
        2                 OD3       Hussain       Food Grains   
        3                 OD4       Jackson  Fruits & Veggies   
        4                 OD5       Ridhesh       Food Grains   
Batch_2 5                 OD6        Adavan       Food Grains   
        6                 OD7         Jonas  Fruits & Veggies   
        7                 OD8         Hafiz  Fruits & Veggies   
        8                 OD9         Hafiz            Bakery   
        9                OD10      Krithika            Bakery   

                          Sub Category            City Order Date Region  \
Source  Original_Row                                                       
Batch_1 0                      Masalas         Vellore 2017-11-08  North   
        1                Health Drinks     Krishnagiri 2017-11-08  South   
        2                 Atta & Flour      Perambalur 2017-06-12   West   
        3             Fresh Vegetables      Dharmapuri 2016-10-11  South   
        4              Organic Staples            Ooty 2016-10-11  South   
Batch_2 5              Organic Staples      Dharmapuri 2015-06-09   West   
        6             Fresh Vegetables          Trichy 2015-06-09   West   
        7                 Fresh Fruits  Ramanadhapuram 2015-06-09   West   
        8                     Biscuits     Tirunelveli 2015-06-09   West   
        9                        Cakes         Chennai 2015-06-09   West   

                      Sales  Discount  Profit       State  
Source  Original_Row                                       
Batch_1 0              1254      0.12  401.28  Tamil Nadu  
        1               749      0.18  149.80  Tamil Nadu  
        2              2360      0.21  165.20  Tamil Nadu  
        3               896      0.25   89.60  Tamil Nadu  
        4              2355      0.26  918.45  Tamil Nadu  
Batch_2 5              2305      0.26  322.70  Tamil Nadu  
        6               826      0.33  346.92  Tamil Nadu  
        7              1847      0.32  147.76  Tamil Nadu  
        8               791      0.23  181.93  Tamil Nadu  
        9              1795      0.27  484.65  Tamil Nadu

Index names: ['Source', 'Original_Row']


## 23. Horizontal concatenation

Use `axis=1` when the two objects contain different columns that refer to the same rows. In this case, we will align two small tables by their reset index.

**Practical use:** this is useful when two reports describe the same observations but contain different attributes and need to be placed side-by-side.


In [21]:
left_part = wrangled[['Order ID', 'Customer Name']].head(8).reset_index(drop=True)
right_part = wrangled[['Category', 'Region', 'Sales']].head(8).reset_index(drop=True)

# Concatenate the two tables side-by-side.
horizontal = pd.concat([left_part, right_part], axis=1)

# Display the result.
display(horizontal)

,Order ID,Customer Name,Category,Region,Sales
0,OD1,Harish,Oil & Masala,North,1254
1,OD2,Sudha,Beverages,South,749
2,OD3,Hussain,Food Grains,West,2360
3,OD4,Jackson,Fruits & Veggies,South,896
4,OD5,Ridhesh,Food Grains,South,2355
5,OD6,Adavan,Food Grains,West,2305
6,OD7,Jonas,Fruits & Veggies,West,826
7,OD8,Hafiz,Fruits & Veggies,West,1847


# Part E – Database-Style Joins with `merge()`

## 24. Why use `merge()`?

A database-style join combines related tables through one or more key columns. Instead of simply stacking rows, it matches records that share a key.

### Why is this useful in practice?

A retail company may keep **sales transactions** in one table and **customer master details** in another. To prepare a customer report, we match the two tables using `Customer Name` or another customer ID.

### Main arguments

- `on=` – same key name in both tables
- `left_on=` / `right_on=` – different key names
- `how=` – `inner`, `left`, `right`, or `outer`
- `indicator=True` – show where each record came from
- `validate=` – check whether the relationship matches the expected join type


## 25. Preparing two different related tables

We will use two tables with **different purposes and different columns**:

- **`customer_summary`** – a transaction-based performance table containing total sales and profit for each customer.
- **`customer_master`** – a separate customer lookup table containing region and customer tier. It also contains a couple of customers who do not yet have transactions, so that unmatched join results are visible.

This is closer to a real reporting workflow than merging two summaries created from exactly the same columns.


In [22]:
# Customer performance table from transaction records.
customer_summary = (
    wrangled.groupby('Customer Name', as_index=False)
           .agg(Total_Sales=('Sales', 'sum'),
                Total_Profit=('Profit', 'sum'))
)

# Separate customer master / lookup table.
# It contains customer attributes rather than sales measures.
master_base = (
    wrangled[['Customer Name', 'Region']]
    .drop_duplicates('Customer Name')
    .sort_values('Customer Name')
    .head(7)
    .copy()
)
master_base['Customer Tier'] = ['Gold', 'Silver', 'Gold', 'Standard', 'Silver', 'Gold', 'Standard'][:len(master_base)]

extra_customers = pd.DataFrame({
    'Customer Name': ['New Customer A', 'New Customer B'],
    'Region': ['East', 'West'],
    'Customer Tier': ['Gold', 'Standard']
})

customer_master = pd.concat([master_base, extra_customers], ignore_index=True)

print('Customer performance table:')
print(customer_summary.head())
print('\nSeparate customer master table:')
print(customer_master)


Customer performance table:
  Customer Name  Total_Sales  Total_Profit
0        Adavan       315341      78044.29
1         Aditi       299674      78439.05
2         Akash       307488      82121.26
3          Alan       294397      74410.23
4        Amrish       333351      80191.89

Separate customer master table:
    Customer Name   Region Customer Tier
0          Adavan     West          Gold
1           Aditi     West        Silver
2           Akash     East          Gold
3            Alan  Central      Standard
4          Amrish  Central        Silver
5             Amy     West          Gold
6             Anu     East      Standard
7  New Customer A     East          Gold
8  New Customer B     West      Standard


## 26. Inner join

An **inner join** keeps only keys that appear in both tables. In this practical, it keeps customers who have both a transaction summary and a matching customer-master record.

**Practical use:** use an inner join when the report should contain only complete matches between two sources.


In [23]:
# Keep only customers present in both the transaction summary and customer master.
inner_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='inner'
)

print(inner_joined)


  Customer Name  Total_Sales  Total_Profit   Region Customer Tier
0        Adavan       315341      78044.29     West          Gold
1         Aditi       299674      78439.05     West        Silver
2         Akash       307488      82121.26     East          Gold
3          Alan       294397      74410.23  Central      Standard
4        Amrish       333351      80191.89  Central        Silver
5           Amy       301558      79159.94     West          Gold
6           Anu       283735      70238.28     East      Standard


## 27. Left join vs Right join – the side matters

The difference is **which table you choose to protect**.

- **Left join:** keep every row from the left table and add matches from the right table.
- **Right join:** keep every row from the right table and add matches from the left table.

Here, the **left table is `customer_summary`** and the **right table is `customer_master`**.

**Practical use:**

- Use a **left join** when your main report is the transaction/customer-performance table and you do not want to lose customers who have sales.
- Use a **right join** when the master table is the list you want to preserve, such as checking all registered customers even if some have no transactions.


In [24]:
# LEFT JOIN – keep all customers who appear in the sales summary.
left_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='left'
)

print('LEFT JOIN – customers missing from the master table:')
print(left_joined[left_joined['Customer Tier'].isna()])

# RIGHT JOIN – keep all customers in the customer master table.
right_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='right'
)

print('\nRIGHT JOIN – customers with no transaction summary:')
print(right_joined[right_joined['Total_Sales'].isna()])


LEFT JOIN – customers missing from the master table:
   Customer Name  Total_Sales  Total_Profit Region Customer Tier
7         Arutra       325720      87572.40    NaN           NaN
8         Arvind       293802      71767.23    NaN           NaN
9         Esther       283656      69588.12    NaN           NaN
10        Ganesh       278349      67094.89    NaN           NaN
11         Hafiz       251965      64037.35    NaN           NaN
12        Harish       293839      79107.83    NaN           NaN
13       Haseena       298924      73167.14    NaN           NaN
14       Hussain       307337      72354.97    NaN           NaN
15       Jackson       271793      70268.59    NaN           NaN
16         James       305915      71926.88    NaN           NaN
17         Jonas       293489      73497.19    NaN           NaN
18         Komal       295671      76165.17    NaN           NaN
19      Krithika       334361      85633.03    NaN           NaN
20         Kumar       262623      67

## 28. Outer join

An **outer join** keeps keys from both tables. It is useful when you want to audit all matched and unmatched records from both sources.

**Practical use:** this is useful during data reconciliation when you need to find records that exist in one system but not the other.


In [25]:
# Keep every customer appearing in either table.
outer_joined = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='outer',
    indicator=True
)

print(outer_joined)


     Customer Name  Total_Sales  Total_Profit   Region Customer Tier  \
0           Adavan     315341.0      78044.29     West          Gold   
1            Aditi     299674.0      78439.05     West        Silver   
2            Akash     307488.0      82121.26     East          Gold   
3             Alan     294397.0      74410.23  Central      Standard   
4           Amrish     333351.0      80191.89  Central        Silver   
5              Amy     301558.0      79159.94     West          Gold   
6              Anu     283735.0      70238.28     East      Standard   
7           Arutra     325720.0      87572.40      NaN           NaN   
8           Arvind     293802.0      71767.23      NaN           NaN   
9           Esther     283656.0      69588.12      NaN           NaN   
10          Ganesh     278349.0      67094.89      NaN           NaN   
11           Hafiz     251965.0      64037.35      NaN           NaN   
12          Harish     293839.0      79107.83      NaN          

## 29. Merging when key names are different

Real datasets often use different names for the same key. Use `left_on=` and `right_on=` instead of forcing the column names to match first.

**Practical use:** one system may use `Customer Name` while another uses `Customer`. The data can still be combined without renaming the original field.


In [26]:
# Rename the key in the master table only for this example.
customer_master_alt = customer_master.rename(columns={'Customer Name': 'Customer'})

# Merge using different key names.
different_key_merge = pd.merge(
    customer_summary,
    customer_master_alt,
    left_on='Customer Name',
    right_on='Customer',
    how='left'
)

different_key_merge.head()


,Customer Name,Total_Sales,Total_Profit,Customer,Region,Customer Tier
0,Adavan,315341,78044.29,Adavan,West,Gold
1,Aditi,299674,78439.05,Aditi,West,Silver
2,Akash,307488,82121.26,Akash,East,Gold
3,Alan,294397,74410.23,Alan,Central,Standard
4,Amrish,333351,80191.89,Amrish,Central,Silver


## 30. Checking the type of merge

The `validate=` argument can check whether the relationship between tables matches your expectation.

For example, `many_to_one` expects repeated keys on the left but a single matching key on the right. That is appropriate here because many transaction rows may belong to one customer, while the customer master contains one record per customer.

**Practical use:** `validate=` helps catch accidental duplicate master records that could multiply rows in a report.


In [27]:
# Check a true many-to-one relationship:
# many transaction rows can belong to one customer-master row.
validated_merge = pd.merge(
    wrangled[['Order ID', 'Customer Name', 'Sales']],
    customer_master,
    on='Customer Name',
    how='left',
    validate='many_to_one'
)

validated_merge.head()


,Order ID,Customer Name,Sales,Region,Customer Tier
0,OD1,Harish,1254,NaN,NaN
1,OD2,Sudha,749,NaN,NaN
2,OD3,Hussain,2360,NaN,NaN
3,OD4,Jackson,896,NaN,NaN
4,OD5,Ridhesh,2355,NaN,NaN


## 31. Tracking unmatched rows with `indicator=True`

When auditing two tables, `indicator=True` creates a column showing whether each row came from the left table, right table, or both.

**Practical use:** this is helpful during reconciliation—for example, when checking which customers exist in the transaction system but are missing from the master system.


In [28]:
# Perform an outer merge with indicator=True.
merge_check = pd.merge(
    customer_summary,
    customer_master,
    on='Customer Name',
    how='outer',
    indicator=True
)

print(merge_check['_merge'].value_counts())


_merge
left_only     43
both           7
right_only     2
Name: count, dtype: int64


## 32. Merging on indexes

Sometimes the key is already stored as the index. `merge()` can work directly with index values by using `left_index=True` and `right_index=True`.

In practice, a regional sales table might be matched with a separate regional target table using the region stored as the index.


In [29]:
# Actual sales from transaction records.
region_sales = wrangled.groupby('Region')['Sales'].sum().to_frame('Actual_Sales')

# Separate business target table.
region_targets = pd.DataFrame({
    'Sales_Target': [18000, 17000, 16000, 15000, 19000],
    'Region_Manager': ['Anil', 'Meera', 'Rohan', 'Priya', 'Karan']
}, index=['Central', 'East', 'North', 'South', 'West'])

# Merge the two different tables using their indexes.
region_report_merge = pd.merge(
    region_sales,
    region_targets,
    left_index=True,
    right_index=True,
    how='left'
)

region_report_merge


,Actual_Sales,Sales_Target,Region_Manager
Region,,,
Central,3468156,18000,Anil
East,4248368,17000,Meera
North,1254,16000,Rohan
South,2440461,15000,Priya
West,4798743,19000,Karan


## 33. Using `join()` for index-based combination

`DataFrame.join()` is convenient when the joining key is already the index. It can make index-based combinations shorter and easier to read.

**Practical use:** once regional targets and actuals are indexed by `Region`, `join()` is often a readable way to build a management report.


In [30]:
# Create another separate lookup table indexed by Region.
region_manager = region_targets[['Region_Manager']]

# Join actual sales to the regional manager lookup.
region_report = region_sales.join(region_manager, how='left')

region_report


,Actual_Sales,Region_Manager
Region,,
Central,3468156,Anil
East,4248368,Meera
North,1254,Rohan
South,2440461,Priya
West,4798743,Karan


## 34. Choosing `merge()` or `join()`

Use `merge()` when:

- the key is stored in ordinary columns
- the left and right key names are different
- you need detailed control over join types

Use `join()` when:

- the key is already the index
- you want a concise index-based combination

**Practical rule:** first ask whether the relationship is a **column-to-column match** or an **index-to-index alignment**. That usually tells you which method to choose.


# Part F – Reshaping and Pivoting

## 35. Why reshape data?

The same information can be stored in different layouts. Wrangling often means changing the layout without changing the underlying business meaning.

### Why is this useful in practice?

A detailed sales table may be best for analysis, while a manager may want **Region as rows and Category as columns** for a report. Reshaping lets the same data move between these forms without rebuilding the data from scratch.


## 36. Create a wide-format sales table

We will first summarise sales by Region and Category. Then we will place categories across columns to create a wide layout.

In [31]:
# Summarise total Sales by Region and Category.
region_category_sales = (
    wrangled.groupby(['Region', 'Category'], as_index=False)['Sales']
           .sum()
)

# Create a wide table with Region as rows and Category as columns.
wide_sales = region_category_sales.pivot(
    index='Region', columns='Category', values='Sales'
)

# Display the wide table.
display(wide_sales)

Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,448343.0,501194.0,516642.0,463683.0,526507.0,453293.0,558494.0
East,588241.0,577206.0,677830.0,598868.0,564237.0,622071.0,619915.0
North,NaN,NaN,NaN,NaN,NaN,1254.0,NaN
South,361385.0,344666.0,360963.0,353069.0,352162.0,298859.0,369357.0
West,714312.0,662247.0,711966.0,699652.0,657821.0,662965.0,689780.0


## 37. Wide-to-long with `melt()`

`melt()` turns multiple value columns into two columns: one column containing the variable name and another containing the value.

**Practical use:** `melt()` is useful when a report is stored in a wide layout but you need a long, tidy layout for grouping, plotting or further analysis.


In [32]:
# Convert wide_sales back to long format.
long_sales = wide_sales.reset_index().melt(
    id_vars='Region',
    var_name='Category',
    value_name='Sales'
)

# Display the result.
display(long_sales)

,Region,Category,Sales
0,Central,Bakery,448343.0
1,East,Bakery,588241.0
2,North,Bakery,NaN
3,South,Bakery,361385.0
4,West,Bakery,714312.0
5,Central,Beverages,501194.0
6,East,Beverages,577206.0
7,North,Beverages,NaN
8,South,Beverages,344666.0
9,West,Beverages,662247.0


## 38. Inspecting the long-format result

After reshaping, always verify the row count, column names and missing values. Reshaping can introduce `NaN` where a Region does not contain a particular category.

In [33]:
# Display the shape of long_sales.
print('Shape:', long_sales.shape)
# Display the column names.
print('Columns:', long_sales.columns.tolist())
# Count missing Sales values.
print('Missing Sales values:', long_sales['Sales'].isna().sum())

Shape: (35, 3)
Columns: ['Region', 'Category', 'Sales']
Missing Sales values: 6


## 39. Long-to-wide with `pivot()`

`pivot()` reverses a simple melt when each combination of index and column values identifies one observation.

**Practical use:** `pivot()` is useful when you want to turn detailed records into a clean report layout and there is exactly one value for each combination.


In [34]:
# Remove missing category combinations before pivoting back.
long_sales_clean = long_sales.dropna(subset=['Sales'])

# Pivot the long data back to Region × Category.
wide_again = long_sales_clean.pivot(
    index='Region', columns='Category', values='Sales'
)

# Display the result.
display(wide_again)

Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,448343.0,501194.0,516642.0,463683.0,526507.0,453293.0,558494.0
East,588241.0,577206.0,677830.0,598868.0,564237.0,622071.0,619915.0
North,NaN,NaN,NaN,NaN,NaN,1254.0,NaN
South,361385.0,344666.0,360963.0,353069.0,352162.0,298859.0,369357.0
West,714312.0,662247.0,711966.0,699652.0,657821.0,662965.0,689780.0


## 40. `pivot_table()` when aggregation is required

Use `pivot_table()` when duplicate combinations need summarisation.

**Practical use:** in retail data there are many transactions for the same Region and Category. `pivot_table()` can combine those transactions using `sum`, `mean`, `count` or another aggregation function to create a report.


In [35]:
# Create a pivot table directly from the transaction-level data.
pivot_sales = pd.pivot_table(
    wrangled,
    index='Region',
    columns='Category',
    values='Sales',
    aggfunc='sum',
    fill_value=0
)

# Display the pivot table.
display(pivot_sales)

Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,448343,501194,516642,463683,526507,453293,558494
East,588241,577206,677830,598868,564237,622071,619915
North,0,0,0,0,0,1254,0
South,361385,344666,360963,353069,352162,298859,369357
West,714312,662247,711966,699652,657821,662965,689780


## 41. Hierarchical columns with `pivot_table()`

`pivot_table()` can place more than one measure in the columns. The result is a **MultiIndex on the columns**.

This is useful when you want Sales, Profit and perhaps average Discount side-by-side for every Category.

In [36]:
# Create a pivot table with two measures: Sales and Profit.
category_report = pd.pivot_table(
    wrangled,
    index='Region',
    columns='Category',
    values=['Sales', 'Profit'],
    aggfunc='sum',
    fill_value=0
)

# Display the report.
display(category_report)

# Inspect the column index names.
print('Column index names:', category_report.columns.names)


Profit                                                            \
Category     Bakery  Beverages Eggs, Meat & Fish Food Grains Fruits & Veggies   
Region                                                                          
Central   109543.91  128017.14         126587.35   111517.05        131699.53   
East      153741.06  144498.17         164465.08   154932.01        141292.78   
North          0.00       0.00              0.00        0.00             0.00   
South      89102.07   84058.78          93728.18    94823.55         92961.26   
West      176134.02  169031.67         182576.61   167890.03        164446.81   

                                   Sales                              \
Category Oil & Masala     Snacks  Bakery Beverages Eggs, Meat & Fish   
Region                                                                 
Central     109261.91  140179.95  448343    501194            516642   
East        155569.22  159847.26  588241    577206            677830   
North          401.28       0.00       0         0                 0   
South        73909.18   94979.87  361385    344666            360963   
West        158753.70  173171.77  714312    662247            711966   

                                                            
Category Food Grains Fruits & Veggies Oil & Masala  Snacks  
Region                                                      
Central       463683           526507       453293  558494  
East          598868           564237       622071  619915  
North              0                0         1254       0  
South         353069           352162       298859  369357  
West          699652           657821       662965  689780

Column index names: [None, 'Category']


## 42. Working with hierarchical columns

When columns form a MultiIndex, selection can be done using tuples or by selecting one level.

For example, `(Sales, 'Bakery')` identifies one measure-category combination.

In [37]:
# Display the top level of the column MultiIndex.
print('Measure level:', category_report.columns.get_level_values(0).unique().tolist())


# Display the second level of the column MultiIndex.
print('Category level:', category_report.columns.get_level_values(1).unique().tolist())

# Select one Sales column from category_report using a tuple.
# Use an available category name so the example is portable.
chosen_category = category_report.columns.get_level_values(1).unique()[0]
print('Sales for category', chosen_category)
display(category_report[('Sales', chosen_category)])

Measure level: ['Profit', 'Sales']
Category level: ['Bakery', 'Beverages', 'Eggs, Meat & Fish', 'Food Grains', 'Fruits & Veggies', 'Oil & Masala', 'Snacks']
Sales for category Bakery


,Sales
,Bakery
Region,
Central,448343
East,588241
North,0
South,361385
West,714312


## 43. `stack()` – columns to rows

`stack()` moves one column level into the row index. It is useful when you need a more detailed, hierarchical row structure.

**Practical use:** a wide management report can be stacked back into a detailed analysis-friendly structure before further filtering or grouping.


In [38]:
# Stack the Category level of the simple pivot table.
stacked_sales = pivot_sales.stack()

# Display the stacked result.
display(stacked_sales)

# Inspect its index names.
print('Index names:', stacked_sales.index.names)

Region   Category         
Central  Bakery               448343
         Beverages            501194
         Eggs, Meat & Fish    516642
         Food Grains          463683
         Fruits & Veggies     526507
         Oil & Masala         453293
         Snacks               558494
East     Bakery               588241
         Beverages            577206
         Eggs, Meat & Fish    677830
         Food Grains          598868
         Fruits & Veggies     564237
         Oil & Masala         622071
         Snacks               619915
North    Bakery                    0
         Beverages                 0
         Eggs, Meat & Fish         0
         Food Grains               0
         Fruits & Veggies          0
         Oil & Masala           1254
         Snacks                    0
South    Bakery               361385
         Beverages            344666
         Eggs, Meat & Fish    360963
         Food Grains          353069
         Fruits & Veggies     352162
         Oil & Masala         298859
         Snacks               369357
West     Bakery               714312
         Beverages            662247
         Eggs, Meat & Fish    711966
         Food Grains          699652
         Fruits & Veggies     657821
         Oil & Masala         662965
         Snacks               689780
dtype: int64

Index names: ['Region', 'Category']


## 44. `unstack()` – index levels to columns

`unstack()` performs the opposite type of movement: an index level is moved into the columns.

**Practical use:** this is useful when a grouped MultiIndex summary needs to become a wide report for presentation or comparison.


In [39]:
# Unstack the second index level from the Region-Category grouped summary.
# Store the result as unstacked_sales.
unstacked_sales = wrangled.groupby(['Region', 'Category'])['Sales'].sum().unstack('Category')
# Display the result.
display(unstacked_sales)

Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,448343.0,501194.0,516642.0,463683.0,526507.0,453293.0,558494.0
East,588241.0,577206.0,677830.0,598868.0,564237.0,622071.0,619915.0
North,NaN,NaN,NaN,NaN,NaN,1254.0,NaN
South,361385.0,344666.0,360963.0,353069.0,352162.0,298859.0,369357.0
West,714312.0,662247.0,711966.0,699652.0,657821.0,662965.0,689780.0


## 45. Stacking and unstacking multiple levels

The operations can be combined when the data has more than one hierarchical level. This is useful for building detailed reports and then returning them to a tabular format.

In [40]:
# Create a Region-Category summary with two measures.
summary_multi = wrangled.groupby(['Region', 'Category'])[['Sales', 'Profit']].sum()

# Stack the measure columns so that Sales and Profit become another level.
# Store the result as stacked_multi.
stacked_multi = summary_multi.stack()


# Use unstack() to bring Category back to columns.
summary_category_wide = stacked_multi.unstack('Category')
display(stacked_multi.head(12))
display(summary_category_wide.head())

Region   Category                 
Central  Bakery             Sales     448343.00
                            Profit    109543.91
         Beverages          Sales     501194.00
                            Profit    128017.14
         Eggs, Meat & Fish  Sales     516642.00
                            Profit    126587.35
         Food Grains        Sales     463683.00
                            Profit    111517.05
         Fruits & Veggies   Sales     526507.00
                            Profit    131699.53
         Oil & Masala       Sales     453293.00
                            Profit    109261.91
dtype: float64

Category           Bakery  Beverages  Eggs, Meat & Fish  Food Grains  \
Region                                                                 
Central Sales   448343.00  501194.00          516642.00    463683.00   
        Profit  109543.91  128017.14          126587.35    111517.05   
East    Sales   588241.00  577206.00          677830.00    598868.00   
        Profit  153741.06  144498.17          164465.08    154932.01   
North   Sales         NaN        NaN                NaN          NaN   

Category        Fruits & Veggies  Oil & Masala     Snacks  
Region                                                     
Central Sales          526507.00     453293.00  558494.00  
        Profit         131699.53     109261.91  140179.95  
East    Sales          564237.00     622071.00  619915.00  
        Profit         141292.78     155569.22  159847.26  
North   Sales                NaN       1254.00        NaN

## 46. Returning reshaped data to ordinary columns

After `stack()` or `unstack()`, the result may still have one or more index levels. `reset_index()` returns those levels to ordinary columns, which is often easier for exporting or merging.

In [41]:
# Reset the index of the final reshaped table.
final_reshaped = summary_category_wide.reset_index()
# Display the column names.
print(final_reshaped.columns.tolist())
display(final_reshaped.head())

['Region', 'level_1', 'Bakery', 'Beverages', 'Eggs, Meat & Fish', 'Food Grains', 'Fruits & Veggies', 'Oil & Masala', 'Snacks']


Category,Region,level_1,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
0,Central,Sales,448343.00,501194.00,516642.00,463683.00,526507.00,453293.00,558494.00
1,Central,Profit,109543.91,128017.14,126587.35,111517.05,131699.53,109261.91,140179.95
2,East,Sales,588241.00,577206.00,677830.00,598868.00,564237.00,622071.00,619915.00
3,East,Profit,153741.06,144498.17,164465.08,154932.01,141292.78,155569.22,159847.26
4,North,Sales,NaN,NaN,NaN,NaN,NaN,1254.00,NaN


## 47. Wide-to-long with `wide_to_long()`

`wide_to_long()` is useful when column names follow a repeated pattern such as `Sales_2015`, `Sales_2016`, and so on.

**Practical use:** this is common when a report stores one year per column but analysis requires one year per row.


In [42]:
# Build a small region-year wide table for the demonstration.
# Drop rows where 'Order Date' is NaT to ensure we can convert Year to integer.
yearly_clean = wrangled.dropna(subset=['Order Date']).copy()
yearly_clean['Year'] = yearly_clean['Order Date'].dt.year.astype(int)

yearly = (
    yearly_clean.groupby(['Region', 'Year'])['Sales']
           .sum()
           .unstack(fill_value=0)
           .reset_index()
)

# Rename the year columns using the pattern Sales_YYYY.
for col in [c for c in yearly.columns if isinstance(c, (int, np.integer))]:
    yearly = yearly.rename(columns={col: f'Sales_{col}'})

# Convert the repeated Sales_YYYY columns to long format with wide_to_long().
yearly_long = pd.wide_to_long(
    yearly, stubnames='Sales', i='Region', j='Year', sep='_', suffix=r'\d+'
).reset_index()

# Display the result.
display(yearly_long.head(10))

,Region,Year,Sales
0,Central,2015,301514
1,East,2015,292020
2,North,2015,0
3,South,2015,205039
4,West,2015,402082
5,Central,2016,275678
6,East,2016,465079
7,North,2016,0
8,South,2016,195478
9,West,2016,300735


# Part G – Comparing the Main Reshaping Operations

| Operation | Typical direction | Best use |
|---|---|---|
| `melt()` | wide → long | turn many value columns into rows |
| `pivot()` | long → wide | reshape when each key combination is unique |
| `pivot_table()` | long → wide | reshape with aggregation |
| `stack()` | columns → index | create hierarchical row structure |
| `unstack()` | index → columns | spread index levels across columns |
| `wide_to_long()` | wide → long | repeated column-name patterns such as year suffixes |

A useful habit is to look at the **shape you have now** and the **shape you need next** before choosing the function.

# Part H – A Small End-to-End Wrangling Workflow

In a real project, the operations are usually connected rather than used separately.

For example, a sales analyst may start with transaction records, create a MultiIndex summary, attach a separate regional lookup, reshape the result into a management-friendly report, and finally verify the structure before sharing it.


## 48. Build a clean working table

In [43]:
# Keep only the fields needed for the reporting task.
report_base = wrangled[['Order ID', 'Customer Name', 'Region', 'Category', 'Order Date', 'Sales', 'Profit']].copy()

# Create Year from Order Date.
report_base['Year'] = report_base['Order Date'].dt.year

# Check the result.
display(report_base.head())

,Order ID,Customer Name,Region,Category,Order Date,Sales,Profit,Year
0,OD1,Harish,North,Oil & Masala,2017-11-08,1254,401.28,2017.0
1,OD2,Sudha,South,Beverages,2017-11-08,749,149.80,2017.0
2,OD3,Hussain,West,Food Grains,2017-06-12,2360,165.20,2017.0
3,OD4,Jackson,South,Fruits & Veggies,2016-10-11,896,89.60,2016.0
4,OD5,Ridhesh,South,Food Grains,2016-10-11,2355,918.45,2016.0


## 49. Create a hierarchical summary

In [44]:
# Summarise Sales and Profit by Year, Region and Category.
report_summary = (
    report_base.groupby(['Year', 'Region', 'Category'])[['Sales', 'Profit']]
               .sum()
)

# Display the first few rows.
display(report_summary.head())

# Inspect the index levels.
print('Index levels:', report_summary.index.names)
print('Number of index levels:', report_summary.index.nlevels)

Sales    Profit
Year   Region  Category                          
2015.0 Central Bakery             43046   9361.76
               Beverages          34074   7726.43
               Eggs, Meat & Fish  43578  11362.69
               Food Grains        24972   5422.66
               Fruits & Veggies   42550  10098.80

Index levels: ['Year', 'Region', 'Category']
Number of index levels: 3


## 50. Add a region lookup table

In [45]:
region_lookup = pd.DataFrame({
    'Region': ['Central', 'East', 'North', 'South', 'West'],
    'Region_Group': ['Core', 'East Zone', 'North Zone', 'South Zone', 'West Zone']
})

# Reset the summary index and merge the region lookup.
report_flat = report_summary.reset_index()
report_enriched = pd.merge(report_flat, region_lookup, on='Region', how='left', validate='many_to_one')

# Display the enriched report.
display(report_enriched.head())

,Year,Region,Category,Sales,Profit,Region_Group
0,2015.0,Central,Bakery,43046,9361.76,Core
1,2015.0,Central,Beverages,34074,7726.43,Core
2,2015.0,Central,"Eggs, Meat & Fish",43578,11362.69,Core
3,2015.0,Central,Food Grains,24972,5422.66,Core
4,2015.0,Central,Fruits & Veggies,42550,10098.80,Core


## 51. Reshape the report for presentation

In [46]:
# Create a wide report with Region as rows and Category as columns.
final_report = pd.pivot_table(
    report_enriched,
    index=['Year', 'Region_Group', 'Region'],
    columns='Category',
    values='Sales',
    aggfunc='sum',
    fill_value=0
)

# Display the final report.
display(final_report)

# Inspect the index and column structure.
print('Index levels:', final_report.index.names)
print('Column levels:', final_report.columns.names)

Category                     Bakery  Beverages  Eggs, Meat & Fish  \
Year   Region_Group Region                                          
2015.0 Core         Central   43046      34074              43578   
       East Zone    East      35380      27660              61629   
       South Zone   South     34254      13266              39254   
       West Zone    West      50181      53675              66640   
2016.0 Core         Central   36770      39022              48216   
       East Zone    East      68463      70115              68919   
       South Zone   South     41250      25473              28616   
       West Zone    West      54170      26422              40844   
2017.0 Core         Central   55689      49532              57188   
       East Zone    East      61601      59169              71543   
       North Zone   North         0          0                  0   
       South Zone   South     37908      39329              32728   
       West Zone    West      62214      75116              80546   
2018.0 Core         Central   52165      64289              65621   
       East Zone    East      67805      75736              88783   
       South Zone   South     45825      39351              47466   
       West Zone    West     122817      88294             106733   

Category                     Food Grains  Fruits & Veggies  Oil & Masala  \
Year   Region_Group Region                                                 
2015.0 Core         Central        24972             42550         36302   
       East Zone    East           32037             36111         56780   
       South Zone   South          16565             29926         36994   
       West Zone    West           61324             65701         52639   
2016.0 Core         Central        50614             28771         32481   
       East Zone    East           67857             70397         63989   
       South Zone   South          28512             26278         16377   
       West Zone    West           39686             61297         43492   
2017.0 Core         Central        42648             39719         63194   
       East Zone    East           63910             68058         58797   
       North Zone   North              0                 0          1254   
       South Zone   South          43850             47627         33020   
       West Zone    West           92180             52083         81629   
2018.0 Core         Central        78149             85834         37140   
       East Zone    East           68597             76736         85497   
       South Zone   South          43240             53686         43936   
       West Zone    West           79185             93087         89578   

Category                     Snacks  
Year   Region_Group Region           
2015.0 Core         Central   76992  
       East Zone    East      42423  
       South Zone   South     34780  
       West Zone    West      51922  
2016.0 Core         Central   39804  
       East Zone    East      55339  
       South Zone   South     28972  
       West Zone    West      34824  
2017.0 Core         Central   70808  
       East Zone    East      71657  
       North Zone   North         0  
       South Zone   South     28910  
       West Zone    West      69362  
2018.0 Core         Central   80083  
       East Zone    East      80275  
       South Zone   South     35127  
       West Zone    West     109571

Index levels: ['Year', 'Region_Group', 'Region']
Column levels: ['Category']


## 52. Verify the wrangled result

After several transformations, always perform a few structural checks. Look at the shape, missing values, duplicate rows, index structure and column structure.

In [64]:
# Display the shape of final_report.
print('Shape:', final_report.shape)
print()
# Count missing values.
print('Missing values per column:\n', final_report.isna().sum())
print()
# Check for duplicate rows.
print('Duplicate rows:', final_report.duplicated().sum())
print()
# Display the index names.
print('Index names:', final_report.index.names)
print()
# Display the column names.
print('Columns:', final_report.columns.tolist())

Shape: (17, 7)

Missing values per column:
 Category
Bakery               0
Beverages            0
Eggs, Meat & Fish    0
Food Grains          0
Fruits & Veggies     0
Oil & Masala         0
Snacks               0
dtype: int64

Duplicate rows: 0

Index names: ['Year', 'Region_Group', 'Region']

Columns: ['Bakery', 'Beverages', 'Eggs, Meat & Fish', 'Food Grains', 'Fruits & Veggies', 'Oil & Masala', 'Snacks']


# Part I – Quick Checks

Before moving to the practice set, remember these quick checks:

```text
Need rows underneath each other?      → pd.concat(..., axis=0)
Need columns side-by-side?            → pd.concat(..., axis=1)
Need match using a key column?         → pd.merge()
Need match using existing indexes?     → merge(..., left_index=True, right_index=True) or join()
Need wide → long?                      → melt(), wide_to_long(), stack()
Need long → wide?                      → pivot(), pivot_table(), unstack()
Need multiple row/column levels?       → MultiIndex, stack(), unstack(), pivot_table()
```

# Practice Set

For each question, try the task yourself first. **No solutions are provided in this student practice section.** Use the worked examples above as a guide and check the structure of your result after each step.


## Question 1 – Create a MultiIndex

Create a new table with `Region` and `Category` as the MultiIndex. Keep `Sales` and `Profit` in the table.

In [65]:
# Write your solution here.
q1_multi = wrangled.set_index(['Region', 'Category'])[['Sales', 'Profit']]
display(q1_multi.head())


Sales  Profit
Region Category                       
North  Oil & Masala       1254  401.28
South  Beverages           749  149.80
West   Food Grains        2360  165.20
South  Fruits & Veggies    896   89.60
       Food Grains        2355  918.45

## Question 2 – MultiIndex selection

Select all records belonging to the first available Region in the MultiIndex.

In [66]:
# Write your solution here.
first_region = q1_multi.index.get_level_values('Region').unique()[0]
q2_selected = q1_multi.xs(first_region, level='Region')
print('Selected Region:', first_region)
display(q2_selected.head())


Selected Region: North


,Sales,Profit
Category,,
Oil & Masala,1254,401.28


## Question 3 – Reorder and sort levels

Swap the two index levels and then sort the result by Category.

In [67]:
# Write your solution here.
q3_reordered = q1_multi.swaplevel('Region', 'Category').sort_index(level='Category')
display(q3_reordered.head())


Sales  Profit
Category Region                
Bakery   Central   1678  318.82
         Central   1663  731.72
         Central   1259  415.47
         Central   1417   99.19
         Central   2111  189.99

## Question 4 – Row-wise concatenation

Split the first 20 rows into two batches and combine them with `concat()`.

In [68]:
# Write your solution here.
first_20 = wrangled.head(20)
batch_a, batch_b = first_20.iloc[:10].copy(), first_20.iloc[10:20].copy()
q4_combined = pd.concat([batch_a, batch_b], axis=0, ignore_index=True)
print('Combined shape:', q4_combined.shape)
display(q4_combined)


Combined shape: (20, 11)


,Order ID,Customer Name,Category,Sub Category,City,Order Date,Region,Sales,Discount,Profit,State
0,OD1,Harish,Oil & Masala,Masalas,Vellore,2017-11-08,North,1254,0.12,401.28,Tamil Nadu
1,OD2,Sudha,Beverages,Health Drinks,Krishnagiri,2017-11-08,South,749,0.18,149.80,Tamil Nadu
2,OD3,Hussain,Food Grains,Atta & Flour,Perambalur,2017-06-12,West,2360,0.21,165.20,Tamil Nadu
3,OD4,Jackson,Fruits & Veggies,Fresh Vegetables,Dharmapuri,2016-10-11,South,896,0.25,89.60,Tamil Nadu
4,OD5,Ridhesh,Food Grains,Organic Staples,Ooty,2016-10-11,South,2355,0.26,918.45,Tamil Nadu
5,OD6,Adavan,Food Grains,Organic Staples,Dharmapuri,2015-06-09,West,2305,0.26,322.70,Tamil Nadu
6,OD7,Jonas,Fruits & Veggies,Fresh Vegetables,Trichy,2015-06-09,West,826,0.33,346.92,Tamil Nadu
7,OD8,Hafiz,Fruits & Veggies,Fresh Fruits,Ramanadhapuram,2015-06-09,West,1847,0.32,147.76,Tamil Nadu
8,OD9,Hafiz,Bakery,Biscuits,Tirunelveli,2015-06-09,West,791,0.23,181.93,Tamil Nadu
9,OD10,Krithika,Bakery,Cakes,Chennai,2015-06-09,West,1795,0.27,484.65,Tamil Nadu


## Question 5 – Database-style left join

Create a customer sales summary and merge it with the **separate `customer_master` table** using a left join. Check that all customers from the sales summary are retained.


In [69]:
# Write your solution here.
q5_summary = wrangled.groupby('Customer Name', as_index=False).agg(Total_Sales=('Sales', 'sum'), Total_Profit=('Profit', 'sum'))
q5_left = pd.merge(q5_summary, customer_master, on='Customer Name', how='left')
print('Summary customers:', len(q5_summary), '| Rows after left join:', len(q5_left))
display(q5_left.head())


Summary customers: 50 | Rows after left join: 50


,Customer Name,Total_Sales,Total_Profit,Region,Customer Tier
0,Adavan,315341,78044.29,West,Gold
1,Aditi,299674,78439.05,West,Silver
2,Akash,307488,82121.26,East,Gold
3,Alan,294397,74410.23,Central,Standard
4,Amrish,333351,80191.89,Central,Silver


## Question 6 – Compare with a right join and different key names

First perform a right join so that all customers in `customer_master` are retained. Then rename `Customer Name` in the master table to `Customer` and repeat the merge using `left_on` and `right_on`.


In [70]:
# Write your solution here.
# Right join retains every row in customer_master, including customers without sales.
q6_right = pd.merge(customer_summary, customer_master, on='Customer Name', how='right')

# Different key names can still match using left_on and right_on.
customer_master_renamed = customer_master.rename(columns={'Customer Name': 'Customer'})
q6_different_keys = pd.merge(customer_summary, customer_master_renamed, left_on='Customer Name', right_on='Customer', how='left')
print('Right join:')
display(q6_right)
print('Different key names:')
display(q6_different_keys.head())


Right join:


,Customer Name,Total_Sales,Total_Profit,Region,Customer Tier
0,Adavan,315341.0,78044.29,West,Gold
1,Aditi,299674.0,78439.05,West,Silver
2,Akash,307488.0,82121.26,East,Gold
3,Alan,294397.0,74410.23,Central,Standard
4,Amrish,333351.0,80191.89,Central,Silver
5,Amy,301558.0,79159.94,West,Gold
6,Anu,283735.0,70238.28,East,Standard
7,New Customer A,NaN,NaN,East,Gold
8,New Customer B,NaN,NaN,West,Standard


Different key names:


,Customer Name,Total_Sales,Total_Profit,Customer,Region,Customer Tier
0,Adavan,315341,78044.29,Adavan,West,Gold
1,Aditi,299674,78439.05,Aditi,West,Silver
2,Akash,307488,82121.26,Akash,East,Gold
3,Alan,294397,74410.23,Alan,Central,Standard
4,Amrish,333351,80191.89,Amrish,Central,Silver


## Question 7 – Merge on indexes

Create a Region-level Sales table and a separate Region-level target table, then combine them using `left_index=True` and `right_index=True`.


In [71]:
# Write your solution here.
q7_sales = wrangled.groupby('Region')['Sales'].sum().to_frame('Actual_Sales')
q7_targets = region_targets[['Sales_Target']]

q7_report = pd.merge(q7_sales, q7_targets, left_index=True, right_index=True, how='left')
display(q7_report)


,Actual_Sales,Sales_Target
Region,,
Central,3468156,18000
East,4248368,17000
North,1254,16000
South,2440461,15000
West,4798743,19000


## Question 8 – Wide to long with `melt()`

Create a Region × Category Sales table and convert it to long format.

In [72]:
# Write your solution here.
q8_wide = pd.pivot_table(wrangled, index='Region', columns='Category', values='Sales', aggfunc='sum')
q8_long = q8_wide.reset_index().melt(id_vars='Region', var_name='Category', value_name='Sales').dropna(subset=['Sales'])
display(q8_long.head(12))


,Region,Category,Sales
0,Central,Bakery,448343.0
1,East,Bakery,588241.0
3,South,Bakery,361385.0
4,West,Bakery,714312.0
5,Central,Beverages,501194.0
6,East,Beverages,577206.0
8,South,Beverages,344666.0
9,West,Beverages,662247.0
10,Central,"Eggs, Meat & Fish",516642.0
11,East,"Eggs, Meat & Fish",677830.0


## Question 9 – Long to wide with `pivot_table()`

Use `pivot_table()` to create a Region × Category table showing total Profit.

In [73]:
# Write your solution here.
q9_profit_wide = pd.pivot_table(wrangled, index='Region', columns='Category', values='Profit', aggfunc='sum', fill_value=0)
display(q9_profit_wide)


Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,109543.91,128017.14,126587.35,111517.05,131699.53,109261.91,140179.95
East,153741.06,144498.17,164465.08,154932.01,141292.78,155569.22,159847.26
North,0.00,0.00,0.00,0.00,0.00,401.28,0.00
South,89102.07,84058.78,93728.18,94823.55,92961.26,73909.18,94979.87
West,176134.02,169031.67,182576.61,167890.03,164446.81,158753.70,173171.77


## Question 10 – Hierarchical columns

Create a pivot table containing both Sales and Profit by Region and Category. Inspect its MultiIndex columns.

In [74]:
# Write your solution here.
q10_hierarchical = pd.pivot_table(wrangled, index='Region', columns='Category', values=['Sales', 'Profit'], aggfunc='sum', fill_value=0)
display(q10_hierarchical)
print('Column level names:', q10_hierarchical.columns.names)


Profit                                                            \
Category     Bakery  Beverages Eggs, Meat & Fish Food Grains Fruits & Veggies   
Region                                                                          
Central   109543.91  128017.14         126587.35   111517.05        131699.53   
East      153741.06  144498.17         164465.08   154932.01        141292.78   
North          0.00       0.00              0.00        0.00             0.00   
South      89102.07   84058.78          93728.18    94823.55         92961.26   
West      176134.02  169031.67         182576.61   167890.03        164446.81   

                                   Sales                              \
Category Oil & Masala     Snacks  Bakery Beverages Eggs, Meat & Fish   
Region                                                                 
Central     109261.91  140179.95  448343    501194            516642   
East        155569.22  159847.26  588241    577206            677830   
North          401.28       0.00       0         0                 0   
South        73909.18   94979.87  361385    344666            360963   
West        158753.70  173171.77  714312    662247            711966   

                                                            
Category Food Grains Fruits & Veggies Oil & Masala  Snacks  
Region                                                      
Central       463683           526507       453293  558494  
East          598868           564237       622071  619915  
North              0                0         1254       0  
South         353069           352162       298859  369357  
West          699652           657821       662965  689780

Column level names: [None, 'Category']


## Question 11 – Stack and unstack

Stack the Category level of a Region × Category Sales table and then unstack it back.

In [75]:
# Write your solution here.
q11_wide = pd.pivot_table(wrangled, index='Region', columns='Category', values='Sales', aggfunc='sum', fill_value=0)
q11_stacked = q11_wide.stack()
q11_restored = q11_stacked.unstack()
display(q11_stacked.head(12))
display(q11_restored)


Region   Category         
Central  Bakery               448343
         Beverages            501194
         Eggs, Meat & Fish    516642
         Food Grains          463683
         Fruits & Veggies     526507
         Oil & Masala         453293
         Snacks               558494
East     Bakery               588241
         Beverages            577206
         Eggs, Meat & Fish    677830
         Food Grains          598868
         Fruits & Veggies     564237
dtype: int64

Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region,,,,,,,
Central,448343,501194,516642,463683,526507,453293,558494
East,588241,577206,677830,598868,564237,622071,619915
North,0,0,0,0,0,1254,0
South,361385,344666,360963,353069,352162,298859,369357
West,714312,662247,711966,699652,657821,662965,689780


## Question 12 – Hierarchical reshaping

Create a Region-Category summary with Sales and Profit, then move Category from the index to the columns.

In [76]:
# Write your solution here.
q12_grouped = wrangled.groupby(['Region', 'Category'])[['Sales', 'Profit']].sum()
q12_wide = q12_grouped.unstack('Category')
display(q12_wide.head())


Sales                                                           \
Category    Bakery Beverages Eggs, Meat & Fish Food Grains Fruits & Veggies   
Region                                                                        
Central   448343.0  501194.0          516642.0    463683.0         526507.0   
East      588241.0  577206.0          677830.0    598868.0         564237.0   
North          NaN       NaN               NaN         NaN              NaN   
South     361385.0  344666.0          360963.0    353069.0         352162.0   
West      714312.0  662247.0          711966.0    699652.0         657821.0   

                                    Profit                               \
Category Oil & Masala    Snacks     Bakery  Beverages Eggs, Meat & Fish   
Region                                                                    
Central      453293.0  558494.0  109543.91  128017.14         126587.35   
East         622071.0  619915.0  153741.06  144498.17         164465.08   
North          1254.0       NaN        NaN        NaN               NaN   
South        298859.0  369357.0   89102.07   84058.78          93728.18   
West         662965.0  689780.0  176134.02  169031.67         182576.61   

                                                               
Category Food Grains Fruits & Veggies Oil & Masala     Snacks  
Region                                                         
Central    111517.05        131699.53    109261.91  140179.95  
East       154932.01        141292.78    155569.22  159847.26  
North            NaN              NaN       401.28        NaN  
South       94823.55         92961.26     73909.18   94979.87  
West       167890.03        164446.81    158753.70  173171.77

## Question 13 – Wide-to-long pattern

Create yearly regional sales columns such as `Sales_2015`, `Sales_2016`, and convert them back to long format with `wide_to_long()`.

In [77]:
# Write your solution here.
q13_wide = yearly.copy()
q13_long = pd.wide_to_long(q13_wide, stubnames='Sales', i='Region', j='Year', sep='_', suffix='\\d+').reset_index()
display(q13_long.head(12))


,Region,Year,Sales
0,Central,2015,301514
1,East,2015,292020
2,North,2015,0
3,South,2015,205039
4,West,2015,402082
5,Central,2016,275678
6,East,2016,465079
7,North,2016,0
8,South,2016,195478
9,West,2016,300735


## Question 14 – Mini case study

Build a report showing total Sales and Profit by Year and Region, add the region group, and create a wide Category report.

In [78]:
# Write your solution here.
q14_summary = wrangled.assign(Year=wrangled['Order Date'].dt.year).groupby(['Year', 'Region', 'Category'], as_index=False)[['Sales', 'Profit']].sum()
q14_enriched = q14_summary.merge(region_lookup, on='Region', how='left', validate='many_to_one')
q14_report = pd.pivot_table(q14_enriched, index=['Year', 'Region', 'Region_Group'], columns='Category', values='Sales', aggfunc='sum', fill_value=0)
display(q14_report.head(15))

Category                     Bakery  Beverages  Eggs, Meat & Fish  \
Year   Region  Region_Group                                         
2015.0 Central Core           43046      34074              43578   
       East    East Zone      35380      27660              61629   
       South   South Zone     34254      13266              39254   
       West    West Zone      50181      53675              66640   
2016.0 Central Core           36770      39022              48216   
       East    East Zone      68463      70115              68919   
       South   South Zone     41250      25473              28616   
       West    West Zone      54170      26422              40844   
2017.0 Central Core           55689      49532              57188   
       East    East Zone      61601      59169              71543   
       North   North Zone         0          0                  0   
       South   South Zone     37908      39329              32728   
       West    West Zone      62214      75116              80546   
2018.0 Central Core           52165      64289              65621   
       East    East Zone      67805      75736              88783   

Category                     Food Grains  Fruits & Veggies  Oil & Masala  \
Year   Region  Region_Group                                                
2015.0 Central Core                24972             42550         36302   
       East    East Zone           32037             36111         56780   
       South   South Zone          16565             29926         36994   
       West    West Zone           61324             65701         52639   
2016.0 Central Core                50614             28771         32481   
       East    East Zone           67857             70397         63989   
       South   South Zone          28512             26278         16377   
       West    West Zone           39686             61297         43492   
2017.0 Central Core                42648             39719         63194   
       East    East Zone           63910             68058         58797   
       North   North Zone              0                 0          1254   
       South   South Zone          43850             47627         33020   
       West    West Zone           92180             52083         81629   
2018.0 Central Core                78149             85834         37140   
       East    East Zone           68597             76736         85497   

Category                     Snacks  
Year   Region  Region_Group          
2015.0 Central Core           76992  
       East    East Zone      42423  
       South   South Zone     34780  
       West    West Zone      51922  
2016.0 Central Core           39804  
       East    East Zone      55339  
       South   South Zone     28972  
       West    West Zone      34824  
2017.0 Central Core           70808  
       East    East Zone      71657  
       North   North Zone         0  
       South   South Zone     28910  
       West    West Zone      69362  
2018.0 Central Core           80083  
       East    East Zone      80275

# Final Challenge – Build a Wrangled Reporting Dataset

Create your own report from the raw transaction data. The final result should:

1. use a MultiIndex at one intermediate stage,
2. combine at least two tables with `concat()` or `merge()`,
3. perform at least one index-based combination,
4. create a wide report with `pivot_table()`,
5. demonstrate either `stack()` or `unstack()`, and
6. finish with a verification section showing shape, missing values and index/column structure.

Try to keep the final report readable enough that another analyst can understand it without reading every intermediate cell.

In [79]:
# Build your own end-to-end wrangling pipeline here.
# Keep the intermediate tables so that each transformation can be checked.
# End-to-end pipeline: clean a working copy, create a MultiIndex summary, combine lookup data, and reshape for reporting.
# Step 1: working copy and datetime/year cleanup.
final_working = sales.copy()
final_working['Order Date'] = pd.to_datetime(final_working['Order Date'], errors='coerce')
final_working = final_working.dropna(subset=['Order Date', 'Region', 'Category', 'Sales', 'Profit'])
final_working['Year'] = final_working['Order Date'].dt.year

# Step 2: MultiIndex intermediate summary for auditability.
challenge_summary = final_working.groupby(['Region', 'Category'])[['Sales', 'Profit']].sum()

# Step 3: combine regional lookup information with merge; validate lookup uniqueness.
challenge_flat = challenge_summary.reset_index()
challenge_lookup = pd.DataFrame({'Region': ['Central', 'East', 'North', 'South', 'West'], 'Region_Group': ['Core', 'East Zone', 'North Zone', 'South Zone', 'West Zone']})
challenge_enriched = challenge_flat.merge(challenge_lookup, on='Region', how='left', validate='many_to_one')

# Step 4: combine two independent batches with concat (demonstrates row-wise concatenation).
challenge_batches = pd.concat([final_working.head(5), final_working.iloc[5:10]], ignore_index=True)

# Step 5: create a report with hierarchical row index and category columns.
challenge_report = pd.pivot_table(challenge_enriched, index=['Region_Group', 'Region'], columns='Category', values='Sales', aggfunc='sum', fill_value=0)

# Step 6: verify structure and retain intermediate objects for inspection.
print('MultiIndex summary:')
display(challenge_summary.head())
print('Concatenated sample shape:', challenge_batches.shape)
print('Final report shape:', challenge_report.shape)
print('Missing values:', int(challenge_report.isna().sum().sum()))
display(challenge_report)

MultiIndex summary:


Sales    Profit
Region  Category                           
Central Bakery             187670  42730.17
        Beverages          186917  45775.03
        Eggs, Meat & Fish  214603  51821.67
        Food Grains        196383  48181.53
        Fruits & Veggies   196874  48119.45

Concatenated sample shape: (10, 12)
Final report shape: (5, 7)
Missing values: 0


,Category,Bakery,Beverages,"Eggs, Meat & Fish",Food Grains,Fruits & Veggies,Oil & Masala,Snacks
Region_Group,Region,,,,,,,
Core,Central,187670,186917,214603,196383,196874,169117,267687
East Zone,East,233249,232680,290874,232401,251302,265063,249694
North Zone,North,0,0,0,0,0,1254,0
South Zone,South,159237,117419,148064,132167,157517,130327,127789
West Zone,West,289382,243507,294763,272375,272168,267338,265679


# Quick Reference

| Function | Purpose |
|---|---|
| `set_index()` | create index / MultiIndex from columns |
| `pd.MultiIndex.from_arrays()` | create a MultiIndex from arrays |
| `pd.MultiIndex.from_tuples()` | create a MultiIndex from tuples |
| `.loc[]` | label-based selection |
| `xs()` | select a cross-section from one index level |
| `swaplevel()` | exchange MultiIndex levels |
| `reorder_levels()` | change the level order |
| `sort_index()` | sort by index levels |
| `reset_index()` | move index levels back to columns |
| `pd.concat()` | combine tables along rows or columns |
| `pd.merge()` | database-style joins (`inner`, `left`, `right`, `outer`) |
| `DataFrame.join()` | convenient index-based join |
| `melt()` | wide → long |
| `pivot()` | long → wide without aggregation |
| `pivot_table()` | long → wide with aggregation |
| `stack()` | columns → index |
| `unstack()` | index → columns |
| `wide_to_long()` | repeated wide-column patterns → long |

### Key idea

**Data wrangling is mainly about changing structure without losing meaning.** Before choosing a function, ask: *What is the key? What should stay as rows? What should become columns? Do I need matching, stacking, or aggregation?*

## End of Practical

You have now worked through the main pandas operations used for hierarchical indexing, combining datasets, database-style joins, merging on indexes, reshaping, and pivoting. Keep the intermediate tables while practising—you will often learn more from comparing the structures before and after a transformation than from looking only at the final result.